**Menjalankan SparkSession**

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


**A. EXTRACT** *(bobot 10%)*

Baca ketiga sumber data **dari HDFS** (bukan disk lokal) menjadi tiga Spark DataFrame.


In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Tugas7-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Membaca data dari HDFS
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_transaksi.csv",
    header=True,
    inferSchema=True
)

df_cabang = spark.read.json(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_cabang.json"
)

df_menu = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/raw/tugas7_menu.csv",
    header=True,
    inferSchema=True
)

print("Jumlah transaksi :", df_transaksi.count())
print("Jumlah cabang    :", df_cabang.count())
print("Jumlah menu      :", df_menu.count())

df_transaksi.show(5)
df_cabang.show()
df_menu.show(5)

Jumlah transaksi : 4000
Jumlah cabang    : 3


Jumlah menu      : 15
+------+---------+-------+---+
|trx_id|cabang_id|menu_id|qty|
+------+---------+-------+---+
|   KN0|        1|     14|  4|
|   KN1|        1|     11|  4|
|   KN2|        1|     11|  3|
|   KN3|        3|     11|  1|
|   KN4|        2|     12|  4|
+------+---------+-------+---+
only showing top 5 rows

+---------+--------------+----------+--------------------+
|cabang_id|kepala_barista|      kota|         nama_cabang|
+---------+--------------+----------+--------------------+
|        1|         Nadia|  Magelang|Kopi Nusantara Ma...|
|        2|          Reza|Yogyakarta|Kopi Nusantara Yo...|
|        3|         Nadia|  Semarang|Kopi Nusantara Se...|
+---------+--------------+----------+--------------------+

+-------+---------+--------------+-----+
|menu_id|nama_menu| kategori_menu|harga|
+-------+---------+--------------+-----+
|      1|   Menu-1|Makanan Ringan|35000|
|      2|   Menu-2|Makanan Ringan|22000|
|      3|   Menu-3|Makanan Ringan|30000|
|      4|   Me

**B. TRANSFORM — Join & Kolom Turunan** *(bobot 20%)*

Gabungkan ketiga tabel, tambahkan kolom `total_penjualan` (`qty x harga`).

In [9]:
df_gabungan = df_transaksi \
    .join(df_cabang, on="cabang_id", how="inner") \
    .join(df_menu, on="menu_id", how="inner")

df_gabungan = df_gabungan.withColumn(
    "total_penjualan",
    col("qty") * col("harga")
)

df_gabungan.show(10)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


**C. TRANSFORM — Window Function** *(bobot 20%)*

Tentukan **top-2 menu terlaris** (berdasarkan `total_penjualan`) di **setiap cabang**, gunakan `row_number()`.

In [10]:
# Total penjualan setiap menu pada masing-masing cabang
ringkasan_menu = df_gabungan.groupBy(
    "nama_cabang",
    "nama_menu"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan")
)

# Window berdasarkan cabang
window_cabang = Window \
    .partitionBy("nama_cabang") \
    .orderBy(col("total_penjualan").desc())

# Memberikan peringkat
top2_menu = ringkasan_menu.withColumn(
    "peringkat",
    row_number().over(window_cabang)
).filter(
    col("peringkat") <= 2
)

top2_menu.orderBy(
    "nama_cabang",
    "peringkat"
).show(20)

[Stage 40:>                                                         (0 + 1) / 1]

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



**D. TRANSFORM — Spark SQL** *(bobot 20%)*

Menggunakan **Spark SQL murni**, tampilkan total penjualan per `kepala_barista`, diurutkan dari tertinggi.

In [11]:
df_gabungan.createOrReplaceTempView("data_kopi")

In [12]:
penjualan_barista = spark.sql("""
    SELECT
        kepala_barista,
        SUM(total_penjualan) AS total_penjualan
    FROM data_kopi
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""")

penjualan_barista.show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



**E. LOAD** *(bobot 15%)*

Simpan data gabungan (hasil bagian B) ke HDFS sebagai Parquet, path `/user/[username]/tugas7/processed/laporan`, dipartisi berdasarkan `kategori_menu`. Verifikasi dengan membaca kembali dan menghitung jumlah baris.

In [13]:
!hdfs dfs -mkdir -p /user/mahasiswa/tugas7/processed

In [14]:
df_gabungan.write \
    .mode("overwrite") \
    .partitionBy("kategori_menu") \
    .parquet(
        "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
    )

print("Data berhasil disimpan sebagai Parquet.")

Data berhasil disimpan sebagai Parquet.


In [15]:
df_hasil = spark.read.parquet(
    "hdfs://localhost:9000/user/mahasiswa/tugas7/processed/laporan"
)

print("Jumlah baris hasil:", df_hasil.count())

df_hasil.show(5)

Jumlah baris hasil: 4000
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu|harga|total_penjualan| kategori_menu|
+-------+---------+------+---+--------------+----------+--------------------+---------+-----+---------------+--------------+
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|35000|         140000|Makanan Ringan|
|      9|        3|   KN8|  4|         Nadia|  Semarang|Kopi Nusantara Se...|   Menu-9|35000|         140000|Makanan Ringan|
|      3|        1|  KN10|  1|         Nadia|  Magelang|Kopi Nusantara Ma...|   Menu-3|30000|          30000|Makanan Ringan|
|     13|        2|  KN13|  3|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-13|35000|         105000|Makanan Ringan|
|     12|        3|  KN15|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-12|35000|       

In [16]:
!hdfs dfs -ls /user/mahasiswa/tugas7/processed/laporan

Found 4 items
-rw-r--r--   3 vey supergroup          0 2026-10-04 13:28 /user/mahasiswa/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - vey supergroup          0 2026-10-04 13:28 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - vey supergroup          0 2026-10-04 13:28 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - vey supergroup          0 2026-10-04 13:28 /user/mahasiswa/tugas7/processed/laporan/kategori_menu=Non-Kopi


**F. Ringkasan Eksekutif & Rekomendasi** *(bobot 15%)*

Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (**minimal 120 kata**): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

In [17]:
ringkasan_eksekutif = df_gabungan.groupBy(
    "nama_cabang",
    "kota",
    "kepala_barista"
).agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(
    col("total_penjualan").desc()
)

ringkasan_eksekutif.show()

+--------------------+----------+--------------+---------------+----------------+-------------------------+
|         nama_cabang|      kota|kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+--------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Se...|  Semarang|         Nadia|       90542000|            1393|         64997.8463747308|
|Kopi Nusantara Ma...|  Magelang|         Nadia|       87138000|            1302|        66926.26728110599|
|Kopi Nusantara Yo...|Yogyakarta|          Reza|       84645000|            1305|        64862.06896551724|
+--------------------+----------+--------------+---------------+----------------+-------------------------+



In [18]:
ringkasan_eksekutif.show(10, truncate=False)

+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.06896551724        |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



**Eksplorasi**

In [19]:
eksplorasi_kategori = spark.sql("""
    SELECT
        kategori_menu,
        SUM(qty) AS total_unit_terjual,
        SUM(total_penjualan) AS total_penjualan
    FROM data_kopi
    GROUP BY kategori_menu
    ORDER BY total_unit_terjual DESC
""")

eksplorasi_kategori.show()

+--------------+------------------+---------------+
| kategori_menu|total_unit_terjual|total_penjualan|
+--------------+------------------+---------------+
|Makanan Ringan|              4643|      142757000|
|          Kopi|              3273|       64246000|
|      Non-Kopi|              2023|       55322000|
+--------------+------------------+---------------+



In [20]:
eksplorasi_menu = df_gabungan.groupBy(
    "nama_menu",
    "kategori_menu"
).agg(
    spark_sum("qty").alias("total_qty"),
    spark_sum("total_penjualan").alias("total_penjualan")
).orderBy(
    col("total_penjualan").desc()
).limit(5)

eksplorasi_menu.show()

+---------+--------------+---------+---------------+
|nama_menu| kategori_menu|total_qty|total_penjualan|
+---------+--------------+---------+---------------+
|  Menu-12|Makanan Ringan|      736|       25760000|
|   Menu-9|Makanan Ringan|      673|       23555000|
|   Menu-1|Makanan Ringan|      666|       23310000|
|  Menu-13|Makanan Ringan|      628|       21980000|
|  Menu-15|      Non-Kopi|      719|       21570000|
+---------+--------------+---------+---------------+



In [21]:
spark.stop
print("SparkSession berhasil ditutup")

SparkSession berhasil ditutup
